# Stabilità dei cluster: soggetti e seme di UMAP

I cluster di una run di produzione nascono da una catena: 227.235 voxel per soggetto → UMAP li riassume in 2 coordinate → l'agglomerativo raggruppa i soggetti vicini in quelle coordinate.

Questo notebook chiede: **quanto di quel raggruppamento resiste se si cambia qualcosa nella catena?** Due cose si possono cambiare senza toccare il dato: i soggetti (togliendone una parte) e il seme casuale di UMAP.

- **Legge**: la matrice SDC, la run UMAP di produzione e cinque run di clustering agglomerativo (k = 3, 7, 10, 13, 15).
- **Scrive**: nulla, nessun file viene modificato.
- **Non fa**: non sceglie k e non dà verdetti di "stabile" o "instabile". Produce numeri da leggere insieme.

## Vocabolario

**Mappa** (o *embedding*): le 2 coordinate che UMAP assegna a ogni soggetto. Il clustering non vede i voxel, vede solo questi punti.

**Seme** (`random_state`): il numero che fissa la parte casuale di UMAP. Stesso seme e stessi dati danno sempre la stessa mappa; un seme diverso dà un disegno un po' diverso dello stesso dato. La produzione usa il seme 0.

**Sottocampione**: una parte dei soggetti estratta a caso, qui l'80%, **senza reinserimento** (nessun soggetto compare due volte). Con il reinserimento solo il 62% dei soggetti estratti sarebbe distinto, e le copie a distanza 0 falserebbero i metodi basati sui vicini.

**Run di riferimento**: la run di produzione già salvata. Le sue etichette sono il termine di paragone di ogni confronto.

## Le due misure

**ARI** (*Adjusted Rand Index*) — un numero per confronto, sull'intera suddivisione.

Si guardano tutte le coppie di soggetti: i due stanno nello stesso gruppo in entrambe le suddivisioni, o no? L'ARI misura l'accordo su queste coppie, corretto per il caso: 1 = suddivisioni identiche, circa 0 = l'accordo che darebbe una suddivisione casuale. Dipende da k, quindi confronta ripetizioni dello stesso k e non k diversi fra loro.

**Jaccard** — un numero per ogni gruppo.

Per un gruppo A della run di riferimento si cerca il gruppo B che gli somiglia di più nella suddivisione da confrontare, e si calcola |A ∩ B| / |A ∪ B|: 1 = stessi soggetti, verso 0 = il gruppo si sfalda. Dice *quali* gruppi reggono, cosa che l'ARI da solo non dice.

Nessuna delle due ha una soglia: non esiste un valore oltre il quale un risultato è "stabile".

## I tre esperimenti

Ognuno cambia una cosa diversa della catena.

| | Soggetti | Mappa | Confronto contro |
|---|---|---|---|
| **1a** mappa fissa | 80% | quella di produzione, non rifatta | clustering di riferimento |
| **1b** mappa ricalcolata | 80% | UMAP rifatto sull'80%, seme 0 | clustering di riferimento |
| **2** solo il seme, sull'80% | 80%, gli stessi nelle due mappe | UMAP rifatto due volte, semi 1 e 2 | le due mappe fra loro |
| **3** solo il seme, su tutti | tutti | UMAP rifatto con i semi 1-20 | clustering di riferimento (seme 0) |

1a e 1b si calcolano nello stesso ciclo, e la differenza fra le due dice quanto pesa il rifacimento della mappa: con la mappa fissa UMAP ha già visto anche i soggetti tolti, e le coordinate di chi resta ne portano la traccia, quindi 1a è ottimistica.

Gli esperimenti 2 e 3 isolano il seme: i soggetti non cambiano, cambia solo il disegno.

## Parametri

In [ ]:
import json
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score

ROOT = Path("../..").resolve()
sys.path.insert(0, str(ROOT))

from src.analysis.distances import pairwise_distance
from src.analysis.reduction import REDUCTION_METHODS

# La matrice su cui è stata costruita la produzione, e le run di riferimento
SDC_MATRIX = ROOT / "data/derived/sdc_matrix/16-10_s2.4-vol/matrix.npy"
UMAP_RUN = ROOT / "results/sdc/dim_reduction/production/umap/16-10_s2.4-vol_m_euclidean_nc2"
CLUSTER_RUNS = ROOT / "results/sdc/clustering/production/agglomerative/umap"

# k -> cartella della run di clustering di riferimento
RUNS = {
    3: "26-10_s2.4-vol_m_euclidean_nc2_k3_link_ward_mk_euclidean",
    7: "16-10_s2.4-vol_m_euclidean_nc2_k7_link_ward_mk_euclidean",
    10: "26-10_s2.4-vol_m_euclidean_nc2_k10_link_ward_mk_euclidean",
    13: "16-10_s2.4-vol_m_euclidean_nc2_k13_link_ward_mk_euclidean",
    15: "16-10_s2.4-vol_m_euclidean_nc2_k15_link_ward_mk_euclidean",
}

FRACTION = 0.8             # quota di soggetti tenuta in ogni sottocampione
N_REPEATS = 50             # sottocampioni degli esperimenti 1 e 2
SEED = 0                   # seme del primo sottocampione (poi SEED + 1, SEED + 2, ...)
MAP_SEEDS = (1, 2)         # i due semi di UMAP confrontati nell'esperimento 2
UMAP_SEEDS = range(1, 21)  # i semi di UMAP dell'esperimento 3

## Funzioni

Tutto il calcolo passa da qui.

| Funzione | Cosa fa |
|---|---|
| `params_used` | legge dal `config.md` di una run i parametri con cui è stata eseguita |
| `umap_map` | coordinate UMAP, da un blocco di distanze e un seme |
| `cluster` | etichette dell'agglomerativo sulle coordinate date |
| `subsample` | indici di un sottocampione di soggetti, senza reinserimento |
| `jaccard_per_cluster` | Jaccard di ogni gruppo di riferimento con il suo migliore corrispondente |
| `compare` | ARI e Jaccard fra due suddivisioni degli stessi soggetti |
| `summarize` | media, deviazione standard e minimo di una colonna di risultati |
| `jaccard_table` | i Jaccard per gruppo, con il numero di soggetti del gruppo |
| `plot_ari`, `plot_jaccard` | i due grafici, uguali per tutti gli esperimenti |

`compare` si ferma con un errore se la suddivisione di riferimento ha meno di 2 gruppi: l'ARI darebbe 1 per qualunque confronto, e sembrerebbe un risultato.

In [ ]:
def params_used(run_dir: Path) -> dict:
    """Parametri registrati nel config.md della run (riga "Params used: {...}")."""
    text = (run_dir / "config.md").read_text()
    return json.loads(re.search(r"Params used: (\{.*\})", text).group(1))


def umap_map(distances: np.ndarray, params: dict, seed: int) -> np.ndarray:
    """Coordinate UMAP, calcolate su distanze già pronte invece che sui voxel."""
    return REDUCTION_METHODS["umap"](distances, {**params, "metric": "precomputed", "random_state": seed})


def cluster(coordinates: np.ndarray, params: dict) -> np.ndarray:
    """Etichette dell'agglomerativo: un numero di gruppo per soggetto."""
    return AgglomerativeClustering(**params).fit_predict(coordinates)


def subsample(n_subjects: int, fraction: float, seed: int) -> np.ndarray:
    """Indici di una frazione dei soggetti, estratti senza reinserimento."""
    return np.random.default_rng(seed).choice(n_subjects, round(fraction * n_subjects), replace=False)


def jaccard_per_cluster(reference: np.ndarray, other: np.ndarray) -> dict:
    """Per ogni gruppo di `reference`, il Jaccard con il gruppo di `other` che gli somiglia di più."""
    result = {}
    for c in np.unique(reference):
        in_c = reference == c
        result[int(c)] = max(
            (in_c & (other == d)).sum() / (in_c | (other == d)).sum() for d in np.unique(other)
        )
    return result


def compare(reference: np.ndarray, other: np.ndarray) -> tuple[float, dict]:
    """ARI e Jaccard per gruppo fra due suddivisioni degli stessi soggetti."""
    if len(np.unique(reference)) < 2:
        raise ValueError("the reference partition has fewer than 2 clusters: ARI is undefined")
    return adjusted_rand_score(reference, other), jaccard_per_cluster(reference, other)

In [ ]:
def summarize(results: pd.DataFrame, by: list[str], column: str) -> pd.DataFrame:
    """Media, deviazione standard e minimo di `column`, raggruppati per `by`."""
    return results.groupby(by)[column].agg(["mean", "std", "min"]).round(3)


def jaccard_table(results: pd.DataFrame, sizes: pd.DataFrame) -> pd.DataFrame:
    """Jaccard per gruppo con il numero di soggetti `n`, in ordine di k e di dimensione."""
    return (
        summarize(results, ["k", "cluster"], "jaccard")
        .reset_index()
        .merge(sizes, on=["k", "cluster"])
        .sort_values(["k", "n"], ascending=[True, False])
    )


def plot_ari(curves: dict[str, pd.DataFrame]) -> None:
    """Una curva per voce di `curves`: ARI medio ± deviazione standard contro k."""
    fig, ax = plt.subplots(figsize=(7, 4))
    for (label, results), marker in zip(curves.items(), ("o-", "s--", "^-")):
        point = results.groupby("k")["ari"].agg(["mean", "std"])
        ax.errorbar(point.index, point["mean"], yerr=point["std"], fmt=marker, capsize=3, label=label)
    ax.set_xticks(sorted(set().union(*(set(results["k"]) for results in curves.values()))))
    ax.set_xlabel("k")
    ax.set_ylabel("ARI (media ± dev. standard)")
    ax.set_ylim(0, 1.02)
    ax.legend()
    plt.show()


def plot_jaccard(table: pd.DataFrame) -> None:
    """Un pannello per k: una barra per gruppo (media ± dev. standard), un punto nero sul minimo."""
    ks = sorted(table["k"].unique())
    fig, axes = plt.subplots(1, len(ks), figsize=(4 * len(ks), 3.5), sharey=True)
    for ax, k in zip(axes, ks):
        panel = table[table["k"] == k]
        ax.bar(range(len(panel)), panel["mean"], yerr=panel["std"], capsize=2)
        ax.scatter(range(len(panel)), panel["min"], color="k", s=10, zorder=3)
        ax.set_xticks(range(len(panel)))
        ax.set_xticklabels(panel["n"], rotation=90, fontsize=7)  # l'etichetta è la dimensione del gruppo
        ax.set_title(f"k={k}")
        ax.set_xlabel("gruppi, per numero di soggetti")
    axes[0].set_ylabel("Jaccard (barra: media ± dev. std; punto: minimo)")
    axes[0].set_ylim(0, 1.02)
    plt.show()

## Dati di partenza

Dalla run di riferimento si caricano i parametri di UMAP, le coordinate di produzione e le etichette dei cinque clustering.

Le etichette si confrontano **per posizione**: l'ordine dei soggetti deve quindi essere lo stesso in tutti i file, altrimenti il confronto sarebbe sbagliato senza che niente dia errore. Per questo il controllo sull'ordine, e quello su `n_clusters`, fermano il notebook invece di proseguire su un'ipotesi non verificata.

In [ ]:
umap_params = params_used(UMAP_RUN)
if umap_params["metric"] != "euclidean":
    # Le distanze calcolate più sotto sono euclidee: con un'altra metrica non sarebbero quelle della produzione
    raise ValueError(f"the distance matrix below is euclidean, the UMAP run used {umap_params['metric']!r}")

production_map = np.load(UMAP_RUN / "matrix.npy")  # coordinate di produzione, un soggetto per riga
subject_ids = pd.read_csv(UMAP_RUN / "metadata.csv")["subject_id"]

full_labels, cluster_params = {}, {}
for k, name in RUNS.items():
    meta = pd.read_csv(CLUSTER_RUNS / name / "metadata.csv")
    if not subject_ids.equals(meta["subject_id"]):
        raise ValueError(f"{name}: subject order differs from the UMAP run")
    full_labels[k] = meta["cluster_label"].to_numpy()  # le etichette di riferimento per questo k
    cluster_params[k] = params_used(CLUSTER_RUNS / name)
    if cluster_params[k]["n_clusters"] != k:
        raise ValueError(f"{name}: expected k={k}, found {cluster_params[k]}")

# Numero di soggetti di ogni gruppo di riferimento: serve a leggere i Jaccard per gruppo
sizes = pd.concat(
    [pd.Series(full_labels[k]).value_counts().rename_axis("cluster").reset_index(name="n").assign(k=k) for k in RUNS]
)

n = len(subject_ids)
print(n, "subjects;", umap_params)

## Distanze fra i soggetti

UMAP parte dalla distanza euclidea fra i soggetti, calcolata sui 227.235 voxel. Si calcola **una volta sola** (circa 2 minuti): ogni rifacimento della mappa, su tutti i soggetti o su un sottocampione, riusa un blocco di questa matrice invece di rileggere i 5,3 GB del dato grezzo.

In [ ]:
X = np.load(SDC_MATRIX, mmap_mode="r")  # memoria mappata: le righe si leggono senza caricare 5,3 GB
D = pairwise_distance(X, "euclidean")   # la stessa distanza da cui parte la produzione
print(D.shape, D.dtype)

## Controllo: la catena riproduce la produzione

Prima di confrontare qualunque cosa si verifica che questa catena dia **esattamente** la mappa e i gruppi della produzione. Se non fosse così, ogni confronto più sotto misurerebbe anche la differenza fra le due catene, e non solo l'effetto dei soggetti e del seme.

Attesi: differenza massima 0 fra le due mappe, e ARI 1,0000 per ogni k.

In [ ]:
refit = umap_map(D, umap_params, seed=0)  # stesse distanze, stessi parametri, stesso seme della produzione
print("max |mappa rifatta - mappa di produzione| =", np.abs(refit - production_map).max())

for k in RUNS:
    mine = cluster(production_map, cluster_params[k])
    print(f"k={k}: ARI con le etichette salvate = {adjusted_rand_score(full_labels[k], mine):.4f}")

## Controllo: le funzioni di confronto

`compare` si prova su dati costruiti apposta, dove la risposta è nota: tre nuvole ben separate devono dare ARI e Jaccard vicini a 1, e una suddivisione di riferimento con un solo gruppo deve fermarla con un errore.

In [ ]:
rng = np.random.default_rng(0)
# Tre nuvole di 300 punti, lontane fra loro: qualunque metodo ritrova gli stessi tre gruppi
blobs = np.vstack([rng.normal(c, 0.3, size=(300, 2)) for c in ([0, 0], [10, 0], [0, 10])])
truth = np.repeat([0, 1, 2], 300)

idx = rng.choice(len(blobs), 720, replace=False)
ari, jaccard = compare(truth[idx], cluster(blobs[idx], {"n_clusters": 3, "linkage": "ward"}))
assert ari > 0.99 and min(jaccard.values()) > 0.99, (ari, jaccard)

# Un solo gruppo nel riferimento: deve sollevare, non restituire 1
try:
    compare(np.zeros(10, dtype=int), np.zeros(10, dtype=int))
except ValueError as error:
    print("ok, errore atteso:", error)
else:
    raise AssertionError("single-cluster input must raise")

## Esperimento 1 — togliere soggetti

**Cosa**: 50 sottocampioni dell'80% dei soggetti. Per ognuno si rifà il clustering e lo si confronta con quello di riferimento, ristretto agli stessi soggetti.

**Perché**: se bastano 1169 soggetti in meno a cambiare i gruppi, quei gruppi dipendono da chi è capitato nel campione.

**Come**: due versioni per ogni sottocampione, calcolate nello stesso ciclo.

- *mappa ricalcolata*: UMAP rifatto sul blocco di distanze dei soli soggetti estratti, seme 0 come in produzione;

- *mappa fissa*: le coordinate di produzione dei soggetti estratti, senza rifare UMAP.

Un solo rifacimento di UMAP serve a tutti e cinque i k, perché la mappa non dipende da k. Circa 9 minuti.

In [ ]:
ari_rows, jaccard_rows = [], []

for r in range(N_REPEATS):
    idx = subsample(n, FRACTION, SEED + r)  # gli stessi soggetti per tutti i k di questa ripetizione
    maps = {
        "ricalcolata": umap_map(D[np.ix_(idx, idx)], umap_params, seed=0),  # distanze dei soli estratti
        "fissa": production_map[idx],                                       # coordinate di produzione, non rifatte
    }

    for k in RUNS:
        for map_name, coordinates in maps.items():
            ari, jaccard = compare(full_labels[k][idx], cluster(coordinates, cluster_params[k]))
            ari_rows.append({"repeat": r, "k": k, "map": map_name, "ari": ari})
            jaccard_rows += [
                {"repeat": r, "k": k, "map": map_name, "cluster": c, "jaccard": j} for c, j in jaccard.items()
            ]

ari_df = pd.DataFrame(ari_rows)
jaccard_df = pd.DataFrame(jaccard_rows)

### Risultati: ARI

Una riga per k e versione della mappa, su 50 ripetizioni.

- **mean**: quanto i gruppi si ritrovano, in media, dopo aver tolto il 20% dei soggetti.

- **std**: quanto il risultato cambia da una ripetizione all'altra, cioè quanto dipende da *quali* soggetti sono stati tolti.

- **min**: la ripetizione peggiore.

La riga da leggere come stabilità è quella *ricalcolata*; la *fissa* è un limite superiore.

In [ ]:
summarize(ari_df, ["k", "map"], "ari")

In [ ]:
plot_ari({
    "mappa ricalcolata": ari_df[ari_df["map"] == "ricalcolata"],
    "mappa fissa": ari_df[ari_df["map"] == "fissa"],
})

### Risultati: Jaccard per gruppo

Una riga per gruppo di riferimento, sulla sola mappa ricalcolata, con il numero di soggetti `n` del gruppo.

Un Jaccard medio alto dice che il gruppo ricompare con gli stessi soggetti; basso, che si sfalda. Una deviazione alta dice che a volte ricompare e a volte no. Nei gruppi piccoli il migliore corrispondente si trova più facilmente per caso, quindi vanno letti con più cautela.

Nel grafico la barra di errore può superare 1 perché è media ± deviazione standard, non un valore possibile del Jaccard.

In [ ]:
jaccard_refit = jaccard_table(jaccard_df[jaccard_df["map"] == "ricalcolata"], sizes)
jaccard_refit

In [ ]:
plot_jaccard(jaccard_refit)

## Esperimento 2 — solo il seme, sugli stessi soggetti

**Cosa**: per ognuno dei 50 sottocampioni dell'esperimento 1 si rifà UMAP due volte, con i semi 1 e 2, e si confrontano fra loro i due clustering.

**Perché**: nell'esperimento 1 cambiavano due cose insieme, i soggetti e il disegno della mappa. Qui i soggetti sono gli stessi nelle due mappe, quindi quello che resta è l'effetto del solo seme.

**Come leggerlo**: un ARI vicino a 1 vuol dire che il seme conta poco, e che l'instabilità dell'esperimento 1 viene dai soggetti; un ARI vicino a quello della mappa ricalcolata vuol dire che basta il seme a spiegarla.

I due ARI non hanno lo stesso riferimento (là il clustering completo, qui la seconda mappa), quindi il confronto dà l'ordine di grandezza e non una differenza esatta. Circa 12 minuti.

In [ ]:
map_rows = []

for r in range(N_REPEATS):
    idx = subsample(n, FRACTION, SEED + r)  # gli stessi sottocampioni dell'esperimento 1
    block = D[np.ix_(idx, idx)]
    maps = [umap_map(block, umap_params, seed=s) for s in MAP_SEEDS]  # due disegni degli stessi soggetti

    for k in RUNS:
        labels_a, labels_b = (cluster(m, cluster_params[k]) for m in maps)
        ari, _ = compare(labels_a, labels_b)  # il Jaccard non serve: nessuna delle due mappe è il riferimento
        map_rows.append({"repeat": r, "k": k, "ari": ari})

map_df = pd.DataFrame(map_rows)

### Risultati: effetto del solo seme

ARI fra le due mappe, con accanto quello della mappa ricalcolata dell'esperimento 1 per confronto.

In [ ]:
map_summary = summarize(map_df, ["k"], "ari")
map_summary["ari_ricalcolata_vs_riferimento"] = summarize(
    ari_df[ari_df["map"] == "ricalcolata"], ["k"], "ari"
)["mean"]
map_summary

In [ ]:
plot_ari({
    "solo il seme (stessi soggetti, semi diversi)": map_df,
    "mappa ricalcolata contro il riferimento": ari_df[ari_df["map"] == "ricalcolata"],
})

## Esperimento 3 — solo il seme, su tutti i soggetti

**Cosa**: nessun soggetto tolto. UMAP viene rifatto sui 5845 soggetti con i semi da 1 a 20, e ogni clustering si confronta con quello di riferimento (seme 0).

**Perché**: è la stessa scala della produzione. Dice quanto la suddivisione pubblicata dipende dall'aver usato proprio il seme 0 e, con il Jaccard, *quali* gruppi ricompaiono con ogni seme.

Circa 5 minuti.

In [ ]:
seed_ari_rows, seed_jaccard_rows = [], []

for seed in UMAP_SEEDS:
    embedding = umap_map(D, umap_params, seed)  # tutti i soggetti, un disegno diverso per ogni seme
    for k in RUNS:
        ari, jaccard = compare(full_labels[k], cluster(embedding, cluster_params[k]))
        seed_ari_rows.append({"seed": seed, "k": k, "ari": ari})
        seed_jaccard_rows += [{"seed": seed, "k": k, "cluster": c, "jaccard": j} for c, j in jaccard.items()]

seed_ari_df = pd.DataFrame(seed_ari_rows)
seed_jaccard_df = pd.DataFrame(seed_jaccard_rows)

### Risultati: ARI sui semi

ARI per k sui 20 semi, con accanto i due esperimenti precedenti (stessa misura, ma con il 20% dei soggetti tolti).

In [ ]:
seed_summary = summarize(seed_ari_df, ["k"], "ari")
seed_summary["solo_seme_80%"] = map_summary["mean"]
seed_summary["ricalcolata_80%"] = map_summary["ari_ricalcolata_vs_riferimento"]
seed_summary

### Risultati: Jaccard sui semi

Una riga per gruppo di riferimento: quanto quel gruppo ricompare al variare del seme. La colonna `min` è il caso peggiore sui 20 semi, e nel grafico è il punto nero.

In [ ]:
jaccard_seeds = jaccard_table(seed_jaccard_df, sizes)
jaccard_seeds

In [ ]:
plot_jaccard(jaccard_seeds)

## Limiti

- **La mappa fissa è un limite superiore.** Le coordinate di produzione portano la traccia dei soggetti tolti, quindi sovrastimano la stabilità. Il numero da leggere è quello della mappa ricalcolata.

- **L'ARI dipende da k.** Serve a confrontare ripetizioni dello stesso k, non a dire quale k sia migliore.

- **Il Jaccard usa il migliore corrispondente.** Per un gruppo piccolo è più facile trovarne uno simile per caso.

- **I semi provati sono 20** (esperimento 3) e **2** (esperimento 2), non tutti i semi possibili.

- **Un solo campione di 5845 soggetti.** La stabilità misurata è rispetto alla scelta dei soggetti dentro questo campione, non rispetto a una nuova coorte.

- **Stabile non vuol dire vero**: un gruppo che ricompare sempre non è per questo clinicamente significativo, e uno instabile non è sbagliato — può essere un confine sfumato fra due gruppi vicini.